# 🚩 intraPSIC
## `model_personae.ipynb`

> `model_personae.ipynb`<br>
> Simone J. Skeen x Claude Code (08-05-2026)<br>
> WIP - NOT FOR DISTRIBUTION

Cell ID: df172a73

In [ ]:
%%capture
# === INSTALL DEPENDENCIES === #

%pip install -r ../requirements.txt

# Cell ID: f2e354df

In [ ]:
# === STANDARD LIBRARY IMPORTS === #

import ast
import sys
import warnings
from pathlib import Path

# === THIRD-PARTY IMPORTS === #

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from stepmix.stepmix import StepMix

# IPython display configuration
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# === PANDAS DISPLAY OPTIONS === #
# Show all columns and rows for debugging.

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# === SUPPRESS WARNINGS === #
# Hide FutureWarning and UserWarning to keep output clean.

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

# === LOAD ENVIRONMENT VARIABLES === #
# The .env file should contain KNOW_DIR and optionally OLLAMA_HOST.

load_dotenv(Path('..') / '.env')

# === FILE PATHS === #

DATA_RAW = Path('..') / 'data' / 'raw'
DATA_PROC = Path('..') / 'data' / 'processed'

# Cell ID: 79eeeb98

In [ ]:
# Import labeled dissertation data
d = pd.read_csv(DATA_RAW / 'd_inf_labeled_long.csv', index_col=0)

#d.info()
#d.head(5)

# Cell ID: e0b06eb9

In [ ]:
from condense import condense

d = condense(d)

d.info()
d.head(5)

# Cell ID: e54fe1ff

In [ ]:
#from sanitize import sanitize

#d = sanitize(d)

#d.head(5)

# Cell ID: fc988b11

In [ ]:
# Viz. counts

def plot_pred_counts(df, title='Positive Prediction Counts by Classifier'):
    pred_cols = ['asp', 'dep', 'val', 'prg', 'tgd', 'age', 'race', 'dbty',
                 'sui', 'aut', 'adhd', 'bpd', 'ptsd']
    
    counts = df[pred_cols].sum().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x=counts.index, y=counts.values, ax=ax)
    ax.set_xlabel('Predictor')
    ax.set_ylabel('Count (pred = 1)')
    ax.set_title(title)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_pred_counts(d)

# Cell ID: 50284018

In [ ]:
%%script false --no-raise-error
#############################################################################

# Condense to prob ≥0.90

prefixes = [col.replace('_pred', '') for col in d.columns if col.endswith('_pred')]

for prefix in prefixes:
    pred_col = f'{prefix}_pred'
    prob_col = f'{prefix}_prob'
    d.loc[d[prob_col] < 0.90, pred_col] = 0

d.head(5)
#############################################################################
# Cell ID: 21c72cf8

In [ ]:
# Viz. counts

plot_pred_counts(d)

# Cell ID: 22d7536d

In [ ]:
%%script false --no-raise-error
#############################################################################

# Simulate
N = 10_000
cols = [f'v{i:02d}' for i in range(10)]

d = pd.DataFrame(
    np.random.randint(0, 2, size=(N, len(cols))),
    columns=cols
)

# Inspect & verify
d
#############################################################################
# Cell ID: c4500d79

In [ ]:
for col in d.columns:
    print(col)

In [ ]:
# Subset to manifest varlist

indicators = ['asp', 'dep', 'val', 'tgd', 'prg', 'age', 'race', 'dbty', 
                 'sui', 'aut', 'adhd', 'bpd', 'ptsd']

d_indicators = d[indicators]

d_indicators.head(5)

# Cell ID: 8d58e355

In [ ]:
# Latent class analysis / StepMix intro

### NOTE 8/5: working from the docs, this cell

# Categorical StepMix Model with 3 latent classes
model = StepMix(
    n_components=3, 
    measurement="binary", 
    verbose=1, 
    max_iter=5000, 
    n_init=1, 
    random_state=56,
    )
model.fit(d_indicators)

# Allow missing values
#model_nan = StepMix(n_components=3, measurement="binary_nan", random_state=56)
#model_nan.fit(d)

### NOTE: StepMix supports LatentGOLD-equivalent coviariates of class membership + distal outcomes

### NOTE: excellent model selection tutorial: https://colab.research.google.com/drive/1btXHCx90eCsnUlQv_yN-9AzKDhJP_JkG?usp=drive_link

# Cell ID: cbc9c827

In [ ]:
%%script false --no-raise-error
#############################################################################

%%capture
# Grid search: n_components (1-8)

from tqdm import tqdm

param_grid = {
    'n_components': range(1, 9)
}

results = []

for n_comp in tqdm(param_grid['n_components'], desc='n_components'):
    model = StepMix(
        n_components=n_comp,
        measurement='binary',
        max_iter=5000,
        n_init=20,
        random_state=56,
        verbose=0
    )
    model.fit(d)
    
    results.append({
        'n_components': n_comp,
        'LL': model.score(d) * len(d),
        'AIC': model.aic(d),
        'BIC': model.bic(d),
        'CAIC': model.caic(d),
        'SABIC': model.sabic(d),
        'entropy': model.relative_entropy(d)
    })

#############################################################################
# Cell ID: a045b613

In [ ]:
%%script false --no-raise-error
#############################################################################

# Inspect results
results_df = pd.DataFrame(results)
results_df.sort_values('BIC')

#############################################################################
# Cell ID: 26d75e12